### Data Cleaning and Feature Engineering

### 1. Split timestamp and chat in different columns
To do so, we split each row by keeping the breaking point at M: and save the timestamp into a new dataframe ts. Next, we split our original dataframe df into two columns each for Name [ Name] and respective Conversation [ Convo].

In [3]:
from pandas import DataFrame as df

df = df.iloc[:,0].str.split('M:', 1, expand=True)
ts = df.iloc[:,0].copy()
ts = ts.reset_index(drop = True)
ts.columns = ['Timestamp']
df = df.iloc[:,1].str.split(':', 1, expand=True)
df = df.reset_index(drop = True)
df.columns = ['Name','Convo'] 
              


TypeError: 'property' object is not subscriptable

### 2. Map phone numbers to their person name
Sometimes there are people whose names are not saved in our contact directory and therefore we see their phone numbers during the chat. If needed we can provide them their names in this optional step.

In [5]:
''' for unicode use -> r'12345\xa045555' '''
df['Name'][df['Name'].str.contains(r'12345 45555')] = ' Friend8'
df['Name'][df['Name'].str.contains(r'98765 12222')] = ' Friend9' 
              


TypeError: type 'DataFrame' is not subscriptable

### 3. Removing instances where someone changes their number
It happens most of the time that we come across the following message:

Dean number was changed from +91 55555 55555 to +91 22222 22222. ‎Tap to add.

To remove such instances which are not needed for current analysis, we can use regular expressions and remove any instance(s) which has 2 digits followed by 5 digits and again 5 digits. Of course, you can develop this approach to a next level if any conflict arises with analysis

In [6]:
df = df[df['Name'].str.contains(r'\d{2} \d{5} \d{5}') == False] 
              


TypeError: type 'DataFrame' is not subscriptable

### 4. Remove either of the instance: Image omitted, GIF omitted and video omitted
Since our current motive is to work with text, therefore, we can neglect rows which contained either image, gif or a video.

In [ ]:
df = df[df['Convo'].str.contains(r'image omitted') == False]
df = df[df['Convo'].str.contains(r'video omitted') == False]
df = df[df['Convo'].str.contains(r'GIF omitted') == False]
df = df.reset_index(drop = True) 
              


### Handling Redundancy
We can now arrive at our first outcome of retrieving the topmost 3 words used by every friend from the group. To do so, we first have to select all the unique names from the Name column. This can be done by creating a list of unique names using drop_duplicates() function. Before we proceed with our outcome here is a sample program on drop_duplicates() function.



In [7]:
import numpy as np
import pandas as pd
df = pd.DataFrame([[15, 12, np.nan],
                    [33, np.nan, 54],
                    [10, 32, 96],
                    [np.nan, np.nan, np.nan],
                    [10, 32, 96]])
print(df.drop_duplicates())
#       0     1     2
# 0  15.0  12.0   NaN
# 1  33.0   NaN  54.0
# 2  10.0  32.0  96.0
# 3   NaN   NaN   NaN 
              


      0     1     2
0  15.0  12.0   NaN
1  33.0   NaN  54.0
2  10.0  32.0  96.0
3   NaN   NaN   NaN


Since we now know how to unique values, let's find the top 3 words used by each friend. We start by initializing an empty nested list named all_chat_list and use drop_duplicates() to fetch all unique names. Next, we iterate over all rows and for every individual, we concatenate all of the ones text into a single list and append into all_chat_list.

In [8]:
all_chat_list = []
for i in range(len(df['Name'].drop_duplicates())):
    temp = df['Convo'][df['Name'] == df['Name'].drop_duplicates().reset_index(drop = True)[i]]
    temp = temp.reset_index(drop = True)
    for j in range(1,len(temp)):
        temp[0] += ' ' + temp[j]
    all_chat_list.append(temp[0])
    del temp 
              


KeyError: 'Name'

We can check our result under all_chat_list by visualizing all the chats of Friend7 as shown.

In [9]:
''' Unique Names in order of their appearance in group '''
print(df['Name'].drop_duplicates().reset_index(drop = True))
# 0     Friend1
# 1     Friend3
# 2     Friend8
# 3     Friend2
# 4     Friend9
# 5     Friend5
# 6     Friend6
# 7     Friend7
# 8     Friend4
# Name: Name, dtype: object
''' All chats of Friend7 '''
print(list(all_chat_list)[6])   # Friend7
#             gn gn gn gm gm bye  
 
              


KeyError: 'Name'

To find the frequency of the highest words used by any of a friend we can just simply pass itemfreq from scipy.stats and return N (here 3) numbers of top most used. Given under are 3 topmost words used by Friend2 and Friend3.

In [10]:
from scipy.stats import itemfreq
''' Friend1 Top 3 words '''
fg = itemfreq(list(all_chat_list)[0].split(' '))
fg = fg[fg[:,1].astype(float).argsort()][::-1]
print(fg[1:4])
# [['abroad' '4']
# ['the' '3']
#  ['home' '3']]
''' Friend3 Top 3 words '''
fg = itemfreq(list(all_chat_list)[2].split(' '))
fg = fg[fg[:,1].astype(float).argsort()][::-1]
print(fg[1:4])
# [['company' '3']
#  ["it's" '2']
#  ['hmm' '1']]
 
              


ImportError: cannot import name 'itemfreq' from 'scipy.stats' (/usr/local/python/3.14.2/lib/python3.14/site-packages/scipy/stats/__init__.py)

For advanced text analysis, there is a need of removing stopwords, punctuations etc. which can be done mostly using NLTK* in python.

* For more information on NLTK refer nltk.org

Datetime

Our next task is to find the busiest chatting hour of the group. To do so, we rely on python's datetime library. Let us look at our timestamp dataframe ts.

In [11]:
print(ts.head(5))
# 0    26/08/17, 12:36:23 P
# 1    26/08/17, 12:36:37 P
# 2    26/08/17, 12:36:54 P
# 3    26/08/17, 12:37:08 P
# 4    26/08/17, 12:37:33 P
# Name: 0, dtype: object 
              


NameError: name 'ts' is not defined

As we can observe letter ' M' is missing in the end which is needed to convert the given hour format (12 hours) into 24 hours format. Before, we proceed with appending let us check for any missing value in given dataframe ts.

In [12]:
''' Check which row has missing value '''
print(ts.isnull())
# 0      False
# 1      False
# 2      False
# 3      False
# 4      False
# 5      False
# .       .
# .       .
''' Check total number of non-missing values '''
print(ts.count())
# 122 
              


NameError: name 'ts' is not defined

This verifies that none of the rows contains any missing value, in case if it did then we could have filled the missing value using fillna() function or if needed drop it using dropna(). Next, we can append missing letter ' M' and convert the 12-hour format into 24 hours retaining only hours (excluding minutes and seconds).

In [13]:
import datetime
splitted_ts= ts.str.split(', ')
for i in range(len(ts)):
    splitted_ts[i][1] += 'M'
    temp = datetime.datetime.strptime(splitted_ts[i][1], '%I:%M:%S %p')
    splitted_ts[i][1] = datetime.datetime.strftime(temp, '%H')
 
              


NameError: name 'ts' is not defined

To find the frequency of busiest hour we convert splitted_ts Pandas sequence into a list and pass it into the itemfreq function. Afterwards, we can plot a graph of the hour and its corresponding frequency.

In [14]:
hrs = [ splitted_ts[i][1] for i in range(len(splitted_ts)) ]
hrfreq = itemfreq(hrs)
occ = [float(hrfreq[i][1]) for i in range(len(hrfreq))]
hr = [float(hrfreq[i][0]) for i in range(len(hrfreq))]
plt.plot(hr, occ)
plt.grid('on')
plt.xlabel('24 Hours')
plt.ylabel('Frequency')
plt.title('Frequent chat timings') 
              


NameError: name 'splitted_ts' is not defined

It's clear from the graph that busiest hours of the dummy dataset exist between 10:00 to 13:00 and 20:00 to 22:00.